---
title: "Exercise 7: Annotation"
format: html
editor: source
editor_options: 
  chunk_output_type: console
jupyter: ir
---

In [ ]:
#| warning: false
#| output: false

library(scater)
library(SpatialExperiment)
library(SpatialFeatureExperiment)
library(Voyager)
library(DropletUtils)
library(SingleR)
library(tidyverse)
library(stringr)
library(patchwork)
library(viridis)
library(RColorBrewer)
library(pheatmap)
library(alabaster.sfe)

## Dataset loading
We will start by loading the segmented Visium HD dataset stored in the `SpatialFeatureExperiment` object generated in the previous exercise. It contains information on the non-spatially-aware and spatially-aware clusters computed with graph-based clustering Banksy and PCA.

In [ ]:
# Reload the SpatialFeatureExperiment object if not in the R session already
sfe <- readObject("results/day2/02.3_sfe/")
sfe

To transfer cell-type labels to out Visium HD dataset, we will use a [matching scRNA-seq dataset](https://www.10xgenomics.com/platforms/visium/product-family/dataset-human-crc) generated by the 10X team for the same article. See the record on [GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE280311).

::: callout-important
### Exercise 1
Which technology was used to generate the scRNA-seq dataset and how was the experiment designed?
:::

::: {.callout-tip collapse="true"}
### Answer
This scRNA-seq was generated with the 10X FLEX technology, which is probe-based and allows to work with FFPE tissue blocks.

Sample barcodes, sequenced together with the probe sequences allow to pool multiple samples together and then demultiplex them. Here 2 pools of 2 samples (P2CRC/P4CRC, P3NAT/P5NAT) and 1 pool of 4 samples (P1CRC/P3CRC/P5CRC/P2NAT) were created, so 8 samples in total, and each pool was sequenced on multiple 10X Chromium lanes.
:::

## Download the FLEX dataset

We will download the scRNA-seq data from GEO, but this doesn't include the cell metadata, in particular the annotated cell type that we need here. This is available from Github: https://github.com/10XGenomics/HumanColonCancer_VisiumHD/tree/main

In [ ]:
options(timeout = 1000)

if (!dir.exists("data/FLEX/")) {
  download.file(
    url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE280nnn/GSE280311/suppl/GSE280311_RAW.tar",
    destfile = "data/GSE280311_RAW.tar",
    mode = "wb"
  )
  untar(
    tarfile = "data/GSE280311_RAW.tar",
    exdir = "data/FLEX/"
  )
  file.remove("data/GSE280311_RAW.tar")

  ## Download the cell metadata from 10X Github page
  download.file(
    url = "https://raw.githubusercontent.com/10XGenomics/HumanColonCancer_VisiumHD/refs/heads/main/MetaData/SingleCell_MetaData.csv.gz",
    destfile = "data/FLEX/SingleCell_MetaData.csv.gz",
    mode = "wb"
  )
} else {
  message("Data exists, please proceed to next steps!")
}

## Load the P2CRC sample

The sample matching our Visium HD and Xenium dataset was sequenced on 4 10X channels. For simplicity here, we only load the cell from one channel.

In [ ]:
list.files("data/FLEX/", pattern = "P2CRC") 

ref.sce <- DropletUtils::read10xCounts("data/FLEX/GSM8594533_P1L1_P2CRC_BC1_count_filtered_feature_bc_matrix.h5")
ref.sce
colnames(ref.sce) <- ref.sce$Barcode

rownames(ref.sce) <- uniquifyFeatureNames(
  ID = rowData(ref.sce)$ID,
  names = rowData(ref.sce)$Symbol
)

## We will need the logcounts for the annotation
ref.sce <- logNormCounts(ref.sce)

ref.sce

::: callout-important
### Exercise 2
How many cells were sequenced in this specific run? Why are counts available for only 18,082 genes? What type of information is available in the `colData`?
:::

::: {.callout-tip collapse="true"}
### Answer
We loaded here 10,397 cells from one 10X channel run of the P2CRC sample (multiplexed with another channel).

The FLEX technology is probe-based so it doesn't cover the full transcriptome, mostly the protein-coding genes are targeted. 

So far, there is no information on the cells in the object, expect their 10X barcode.
:::

## Load the annotation

In [ ]:
cell_metadata <- read.csv("data/FLEX/SingleCell_MetaData.csv.gz")
dim(cell_metadata)
head(cell_metadata)

cell_metadata <- cell_metadata |>
  as_tibble() |>
  mutate(Sample = str_extract(Barcode, pattern = "(\\d)$")) |>
  filter(Patient == "P2CRC" & QCFilter == "Keep" & Sample == 1) 
table(colnames(ref.sce) %in% cell_metadata$Barcode) 

ref.sce <- ref.sce[, cell_metadata$Barcode]
colData(ref.sce) <- cbind(colData(ref.sce), cell_metadata)

## Explore the annotated object
data("ditto_colors")
ggcells(ref.sce, aes(x = UMAP1, y=UMAP2, color=Level1)) + 
  geom_point() +
  scale_colour_manual(values = ditto_colors) + 
  theme_bw() 
ggcells(ref.sce, aes(x = UMAP1, y=UMAP2, color=Level2)) + 
  geom_point() +
  scale_colour_manual(values = ditto_colors) + 
  theme_bw() 

::: callout-important
### Exercise 3
What do you observe?
:::

::: {.callout-tip collapse="true"}
### Answer
The cell types match the expectations based on the tissue sample we could study earlier in the course. It includes tumore cells, stromal cells and immune cells.

A few cells in the dataset could be misannotated, although this might be an impression induced by the distortion typical of dimensionality reductions like UMAP.
:::

## Perform the anotation with `SingleR`

To annotate each cell, we use the `SingleR()` method, which uses a correlation-based approach comparing cells from the Visium HD dataset to reference cells from the FLEX dataset with known cell-type annotation.

We will use below the highest-level annotation available. A pseudo-bulk aggregation step before the comparison allows for a better performance. We also disable the fine tuning setp for faster computation here:

In [ ]:
table(colData(ref.sce)$Level1)

# Predict cell-type using SingleR 
pred <- SingleR(test = sfe,
                ref = ref.sce,
                labels = ref.sce$Level1,
                aggr.ref = TRUE,       
                fine.tune = FALSE,     
)
sfe$SingleR_label <- pred$pruned.labels
table(sfe$SingleR_label)

# visualize the predictions on the tissue slide.
plotSpatialFeature(sfe,  
                   colGeometryName = "cellSeg",
                   features = "SingleR_label")

## Or on the UMAP
ggcells(sfe, aes(x = UMAP.1, y=UMAP.2, color=SingleR_label)) + 
  geom_point() +
  scale_colour_manual(values = ditto_colors) + 
  theme_bw() 

:::{.callout-important}
### Exercise 4
What do you think about this annotation? 

How do the proportions of different cell types compare to the annotated scRNA-seq dataset?

Do you identify potentially relevant biological structures? Go back to the calculated clusters and spatial domains, to see how do they match.

Try annotating the dataset with the higher-resolution cell-types lables in `ref.sce$Level2`
:::

:::{.callout-tip collapse="true"}
### Answer
Let's calculate the proportion of cells from each cell type:

In [ ]:
prop.table(table(ref.sce$Level1))
prop.table(table(sfe$SingleR_label))

We observe a relatively higher proportion of Endothelial cells and Fibroblast in the Visium HD. Maybe such cell-types are less easily captured by the droplet-based scRNA-seq technology?

The annotation allows us to identify the different structures of the [intestine mucosa](https://en.wikipedia.org/wiki/Intestinal_villus) very clearly, with the epithelial cells composing the intestinal villi, the smooth muscle layer, and the more scattered immune cells. In this colorectal cancer sample, the invading tumor is identified very clearly. 


Let's compare the annotation to the Leiden clustering for example:

In [ ]:
plotSpatialFeature(sfe,  
                   colGeometryName = "cellSeg",
                   features = "SingleR_label") +
  plotSpatialFeature(sfe,  
                   colGeometryName = "cellSeg",
                   features = "Leiden")

## For a more formal comparison
tab_prop <- prop.table(table(sfe$Leiden, sfe$SingleR_label), margin = 1)
## Display the proportions: round and remove 0s
num_mat <- round(tab_prop, 2)
num_mat[num_mat == 0] <- ""

pheatmap(
  tab_prop,
  cluster_rows = TRUE,
  cluster_cols = TRUE,
  color = viridis(100, option = "magma", direction = -1),
  display_numbers = num_mat,
  main = "Leiden clusters vs. SingleR labels", 
  annotation_colors = list(SingleR_label = setNames(object = ditto_colors[1:9], colnames(tab_prop)))  
)

:::

:::{.callout-important}

## Exercise 5: Xenium (Bonus)

Perform the same annotation task on the Xenium dataset. How do the results compare? What could be a limitation here?
:::

:::{.callout-tip collapse="true"}
### Answer

One limitation of this Xenium dataset is the limited number of genes targeted by the probe panel. Some recent benchmarks seem to suggest that the classical scRNA-seq annotation methods (label-transfer methods) perform relatively well: https://link.springer.com/article/10.1186/s12859-025-06044-0

:::

:::{.callout-important}

## Exercise 6: RCTD (Bonus)

We ignored the fact that segmented data suffer from segmentation error, that multiple cells can overlap on the slide, and that spatial data in general suffers from local diffusion of transcripts. All these result in contamination of each cell's transcriptome with transcripts from other cells. 

It is possible to use deconvolution methods to "decontaminate" the cells. For example the RCTD method from the `spacexr` package ([Cable _et al._ 2022](https://www.nature.com/articles/s41587-021-00830-w)) can be used in flag these cases and provide a principled estimate of the most likely composition. Try to run this method on our dataset, you can refer to [this section of the OSTA book](https://bioconductor.org/books/release/OSTA/pages/seq-deconvolution.html#rctd) for guidance. 

How do you interpret the results?

:::

:::{.callout-tip collapse="true"}
### Answer

See the vignette here: https://raw.githack.com/dmcable/spacexr/master/vignettes/spatial-transcriptomics.html

In [ ]:
## Install latest version from Github
BiocManager::install("dmcable/spacexr")
library(spacexr)
sort(table(ref.sce$Level2))
## We need a minimum of 25 cells for each cell type in the reference, so we'll go for Level1 here
reference <- Reference(counts(ref.sce), 
                       cell_types=setNames(factor(ref.sce$Level1), nm = colnames(ref.sce)), 
                       nUMI=colSums(counts(ref.sce)))
## Create SpatialRNA object
to_annotate <- SpatialRNA(coords=data.frame(spatialCoords(sfe)), 
                          counts=counts(sfe),
                          nUMI=colSums(counts(sfe)))
rctd_data <- create.RCTD(to_annotate, reference)

## Run RCTD
res <- run.RCTD(rctd_data, doublet_mode="doublet")
## We use here the "doublet" mode to fit at most two cell types per segmented cell, probably this is a reasonable assumption for a high-resolution technology such as Visium HD.

## Inspect results
results <- res@results
# normalize the cell type proportions to sum to 1.
norm_weights <- normalize_weights(results$weights) 
cell_type_names <- res@cell_type_info$info[[2]] #list of cell type names

# Plots the confident weights for each cell type as in full_mode (saved as 
# 'results/cell_type_weights.pdf')
plot_weights(cell_type_names, to_annotate, "./results/", norm_weights) 

## Likely "doublets"
sfe$RCTD.doublet <- NA
colData(sfe)[row.names(results$results_df), ]$RCTD.doublet <- as.character(results$results_df$spot_class)
plotSpatialFeature(sfe,  
                   colGeometryName = "cellSeg",
                   features = "RCTD.doublet") 

## Extract major cell type
sfe$RCTD.first <- NA
colData(sfe)[row.names(results$results_df), ]$RCTD.first <- as.character(results$results_df$first_type)
plotSpatialFeature(sfe,  
                   colGeometryName = "cellSeg",
                   features = "RCTD.first") 

## Plotting the normalized weights for each cell type
lapply(colnames(norm_weights), \(k) {
    colData(sfe)$foo <- NA
    colData(sfe)[row.names(norm_weights), ]$foo <- norm_weights[, k]
    plt <- plotSpatialFeature(sfe, features = "foo") + 
      scale_color_gradientn(colors=pals::jet())
    plt + ggtitle(k) + guides(fill="none")
}) |>
  wrap_plots(nrow=3)

<!-- 
Weird vertical line, esp. visible for epithelial, fibroblast... Is this a tile boundary? 
-->

:::


## Saving the final `SpatialFeatureExperiment` object

In [ ]:
alabaster.sfe::saveObject(sfe, "results/day2/02.4_sfe")